# 📚 Python 라이브러리 치트시트
**Pandas / Matplotlib / Seaborn / Scipy / Scikit-learn / Folium / 인과추론**

---

# 1️⃣ Pandas

In [ ]:
import pandas as pd
import numpy as np

# ── 데이터 불러오기 / 저장 ───────────────────
df = pd.read_csv('파일명.csv', encoding='utf-8')
df = pd.read_csv('파일명.csv', encoding='cp949')   # 한글 깨질 때
df = pd.read_excel('파일명.xlsx', sheet_name='Sheet1')
df.to_csv('저장.csv', index=False, encoding='utf-8-sig')  # 엑셀에서 한글 안깨지게
df.to_excel('저장.xlsx', index=False)

# ── 기본 탐색 ───────────────────────────────
df.head()               # 상위 5행
df.tail()               # 하위 5행
df.shape                # (행, 열) 수
df.info()               # 컬럼별 타입, 결측치
df.describe()           # 기술통계
df.describe(include='all')  # 범주형 포함
df.isnull().sum()       # 결측치 개수
df.isnull().mean() * 100  # 결측치 비율(%)
df.duplicated().sum()   # 중복 행 수
df.nunique()            # 컬럼별 고유값 수
df['컬럼'].unique()      # 고유값 목록
df['컬럼'].value_counts()           # 빈도
df['컬럼'].value_counts(normalize=True)  # 비율

# ── 컬럼 선택 / 필터링 ──────────────────────
df['컬럼명']                         # 단일 컬럼
df[['컬럼1', '컬럼2']]               # 다중 컬럼
df.iloc[0:5, 0:3]                    # 인덱스로 선택
df.loc[df['컬럼'] > 10]              # 조건 필터링
df.loc[(df['A'] > 10) & (df['B'] == '값')]  # 다중 조건
df.loc[df['컬럼'].isin(['A', 'B'])]  # 특정 값 포함
df.loc[~df['컬럼'].isin(['A', 'B'])] # 특정 값 제외
df.query('A > 10 and B == "값"')     # query 방식

# ── 결측치 처리 ─────────────────────────────
df.dropna()                           # 결측치 행 삭제
df.dropna(subset=['컬럼1', '컬럼2'])  # 특정 컬럼 기준 삭제
df.fillna(0)                          # 0으로 채우기
df['컬럼'].fillna(df['컬럼'].mean())  # 평균으로
df['컬럼'].fillna(df['컬럼'].median()) # 중앙값으로
df['컬럼'].fillna(method='ffill')     # 앞 값으로 채우기
df['컬럼'].fillna(method='bfill')     # 뒤 값으로 채우기

# ── 데이터 변환 ─────────────────────────────
df['컬럼'] = df['컬럼'].astype('int')
df['컬럼'] = df['컬럼'].astype('float')
df['컬럼'] = df['컬럼'].astype('str')
df['날짜'] = pd.to_datetime(df['날짜'])
df['날짜'] = pd.to_datetime(df['날짜'], format='%Y%m%d')  # 형식 지정
df['연도'] = df['날짜'].dt.year
df['월'] = df['날짜'].dt.month
df['요일'] = df['날짜'].dt.dayofweek  # 0=월 ~ 6=일
df.rename(columns={'기존': '신규'}, inplace=True)
df.drop(columns=['컬럼명'], inplace=True)

# ── 로그 변환 (왜도 줄이기) ─────────────────
df['log_값'] = np.log(df['값'])           # 자연로그
df['log1p_값'] = np.log1p(df['값'])       # log(1+x): 0 포함 데이터에 안전
df['log10_값'] = np.log10(df['값'])       # 상용로그
df['sqrt_값'] = np.sqrt(df['값'])         # 제곱근 변환
# 역변환
df['원래값'] = np.exp(df['log_값'])       # log → 원래값
df['원래값'] = np.expm1(df['log1p_값'])   # log1p → 원래값

# ── 구간화 (binning) ─────────────────────────
df['구간'] = pd.cut(df['나이'], bins=[0,20,40,60,100], labels=['20대미만','20-40','40-60','60+'])
df['분위'] = pd.qcut(df['값'], q=4, labels=['하','중하','중상','상'])  # 4분위

# ── apply / map / 조건부 컬럼 ───────────────
df['새컬럼'] = df['컬럼'].apply(lambda x: x * 2)
df['새컬럼'] = df['컬럼'].map({'A': 1, 'B': 2, 'C': 3})
df['새컬럼'] = np.where(df['값'] > 0, '양수', '음수')  # 조건부 생성
df['새컬럼'] = np.select(
    [df['값'] > 100, df['값'] > 50, df['값'] > 0],
    ['높음', '중간', '낮음'],
    default='음수'
)  # 다중 조건

# ── 그룹연산 ────────────────────────────────
df.groupby('컬럼')['값'].mean()
df.groupby('컬럼')['값'].agg(['mean', 'median', 'std', 'count', 'min', 'max'])
df.groupby(['컬럼1', '컬럼2'])['값'].sum()  # 다중 그룹
df.groupby('컬럼').agg({'값1': 'mean', '값2': 'sum'})  # 컬럼별 다른 집계

# ── 피벗 / 피벗테이블 ───────────────────────
# 피벗 (단순 재구조화)
df.pivot(index='날짜', columns='카테고리', values='값')

# 피벗테이블 (집계 포함)
pd.pivot_table(
    df,
    values='매출',
    index='지역',
    columns='월',
    aggfunc='sum',
    fill_value=0,
    margins=True        # 합계 행/열 추가
)

# 피벗 → 원래대로 (melt)
df_melted = df.melt(id_vars=['날짜'], var_name='카테고리', value_name='값')

# ── 윈도우 함수 ─────────────────────────────
df['이동평균'] = df['값'].rolling(window=7).mean()   # 7일 이동평균
df['누적합'] = df['값'].cumsum()                     # 누적합
df['전일비'] = df['값'].pct_change()                 # 전 행 대비 변화율
df['전일차'] = df['값'].diff()                       # 전 행 대비 차이
df['전년동기'] = df['값'].shift(12)                  # 12행 이전 값

# ── 병합 / 조인 ─────────────────────────────
pd.merge(df1, df2, on='키컬럼', how='left')    # left join
pd.merge(df1, df2, on='키컬럼', how='inner')   # inner join
pd.merge(df1, df2, left_on='A', right_on='B', how='left')  # 컬럼명 다를 때
pd.concat([df1, df2], axis=0, ignore_index=True)  # 행 방향 합치기
pd.concat([df1, df2], axis=1)                      # 열 방향 합치기

# ── 정렬 / 순위 ─────────────────────────────
df.sort_values('컬럼', ascending=False)
df.sort_values(['컬럼1', '컬럼2'], ascending=[True, False])  # 다중 정렬
df['순위'] = df['값'].rank(ascending=False, method='dense')  # 순위
df.nlargest(10, '값')    # 상위 10개
df.nsmallest(10, '값')   # 하위 10개

# ── 문자열 처리 ─────────────────────────────
df['컬럼'].str.strip()                  # 공백 제거
df['컬럼'].str.replace(' ', '_')        # 문자 교체
df['컬럼'].str.contains('키워드')       # 특정 문자 포함 여부
df['컬럼'].str.startswith('A')          # 특정 문자로 시작
df['컬럼'].str.split('-').str[0]        # 분리 후 첫 번째
df['컬럼'].str.len()                    # 문자열 길이
df['컬럼'].str.upper()                  # 대문자
df['컬럼'].str.lower()                  # 소문자

---
# 2️⃣ Matplotlib

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

# ── 한글 폰트 설정 ──────────────────────────
plt.rcParams['font.family'] = 'Malgun Gothic'  # Windows
# plt.rcParams['font.family'] = 'AppleGothic'  # Mac
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100              # 해상도

# ── 기본 구조 ───────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
fig, axes = plt.subplots(2, 3, figsize=(18, 10))  # 2행 3열
fig.suptitle('전체 제목', fontsize=18, fontweight='bold')  # 전체 제목

# ── 주요 그래프 ─────────────────────────────
ax.plot(x, y, color='#2196F3', linestyle='--', linewidth=2, marker='o', label='라벨')
ax.bar(x, y, color='steelblue', alpha=0.7, edgecolor='white')
ax.barh(y, x)                                  # 가로 막대
ax.scatter(x, y, c=color_arr, cmap='viridis', s=50, alpha=0.6)  # 색상 맵 산점도
ax.hist(data, bins=30, density=True, color='green', edgecolor='black')
ax.boxplot([data1, data2, data3], labels=['A','B','C'], patch_artist=True)
ax.pie(values, labels=labels, autopct='%1.1f%%', startangle=90, explode=[0.05]*len(values))
ax.fill_between(x, y1, y2, alpha=0.3)          # 영역 채우기
ax.stackplot(x, y1, y2, y3, labels=['A','B','C'])  # 누적 영역

# ── 꾸미기 ──────────────────────────────────
ax.set_title('제목', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('x축', fontsize=12)
ax.set_ylabel('y축', fontsize=12)
ax.legend(loc='upper right', fontsize=10, framealpha=0.5)
ax.grid(True, alpha=0.3, linestyle='--')
ax.set_xlim(0, 100)
ax.set_ylim(0, 100)
ax.axhline(y=50, color='red', linestyle='--', linewidth=1, label='기준선')  # 수평선
ax.axvline(x=50, color='green', linestyle='--')  # 수직선
ax.annotate('설명', xy=(x, y), xytext=(x+5, y+5),
            arrowprops=dict(arrowstyle='->', color='black'))  # 화살표 주석

# ── 축 포맷 ─────────────────────────────────
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))  # 천단위 콤마
ax.yaxis.set_major_formatter(ticker.PercentFormatter())  # 퍼센트
plt.xticks(rotation=45, ha='right')            # x축 레이블 회전

# ── 색상 팔레트 ─────────────────────────────
colors = plt.cm.tab10.colors          # 기본 10색
cmap = plt.cm.get_cmap('Blues', 5)    # 파란색 계열 5단계

# ── 저장 ────────────────────────────────────
plt.tight_layout()
plt.savefig('그래프.png', dpi=300, bbox_inches='tight', transparent=False)
plt.show()

---
# 3️⃣ Seaborn

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# ── 스타일 설정 ─────────────────────────────
sns.set_theme(style='whitegrid', font='Malgun Gothic', font_scale=1.1)
sns.set_palette('Set2')  # Set1/Set2/pastel/husl/tab10

# ── 분포 시각화 ─────────────────────────────
sns.histplot(data=df, x='컬럼', bins=30, kde=True, hue='그룹')  # 히스토그램
sns.kdeplot(data=df, x='컬럼', fill=True, hue='그룹', common_norm=False)
sns.ecdfplot(data=df, x='컬럼')             # 누적분포
sns.boxplot(data=df, x='카테고리', y='수치', hue='그룹', palette='Set2')
sns.violinplot(data=df, x='카테고리', y='수치', inner='box')  # 내부 박스 포함
sns.stripplot(data=df, x='카테고리', y='수치', jitter=True, alpha=0.5)  # 점 분포

# ── 관계 시각화 ─────────────────────────────
sns.scatterplot(data=df, x='x', y='y', hue='그룹', size='크기변수', style='스타일변수')
sns.lineplot(data=df, x='날짜', y='값', hue='그룹', ci=95)  # 신뢰구간 포함
sns.regplot(data=df, x='x', y='y', ci=95, scatter_kws={'alpha':0.3})  # 회귀선
sns.lmplot(data=df, x='x', y='y', hue='그룹', col='범주')  # 그룹별 회귀
sns.pairplot(df, hue='그룹', diag_kind='kde', plot_kws={'alpha':0.5})
sns.jointplot(data=df, x='x', y='y', kind='hex')  # hex/scatter/kde/reg

# ── 범주형 시각화 ───────────────────────────
sns.barplot(data=df, x='카테고리', y='수치', estimator='mean', errorbar='sd')
sns.countplot(data=df, x='카테고리', order=df['카테고리'].value_counts().index)
sns.pointplot(data=df, x='시점', y='값', hue='그룹')  # 평균 연결선

# ── 히트맵 ──────────────────────────────────
corr = df.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))  # 상삼각 마스크
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, mask=mask,
            linewidths=0.5, square=True)

# ── 클러스터맵 ──────────────────────────────
sns.clustermap(corr, annot=True, fmt='.2f', cmap='coolwarm')  # 계층 클러스터링

# ── FacetGrid (조건별 그래프) ────────────────
g = sns.FacetGrid(df, col='카테고리', row='그룹', height=4)
g.map_dataframe(sns.histplot, x='값')
g.add_legend()

plt.tight_layout()
plt.show()

---
# 4️⃣ Scipy (사회통계 중심)

In [ ]:
from scipy import stats
import numpy as np

# ── 기술통계 ────────────────────────────────
stats.describe(data)               # 기술통계 한번에
stats.zscore(data)                 # z-score 표준화
stats.iqr(data)                    # IQR (사분위범위)
stats.skew(data)                   # 왜도
stats.kurtosis(data)               # 첨도

# ── 정규성 검정 ─────────────────────────────
stat, p = stats.shapiro(data)      # Shapiro-Wilk (n<50 권장)
stat, p = stats.normaltest(data)   # D'Agostino-Pearson (n>20)
stat, p = stats.kstest(data, 'norm', args=(data.mean(), data.std()))  # K-S 검정
# p > 0.05 → 정규분포 O

# ── 등분산 검정 (t검정 전 확인) ─────────────
stat, p = stats.levene(group1, group2)   # Levene 검정
stat, p = stats.bartlett(group1, group2) # Bartlett 검정
# p > 0.05 → 등분산 O

# ── 평균 비교 ───────────────────────────────
stat, p = stats.ttest_1samp(data, popmean=0)             # 단일표본 t검정
stat, p = stats.ttest_ind(g1, g2, equal_var=True)        # 독립표본 t검정 (등분산)
stat, p = stats.ttest_ind(g1, g2, equal_var=False)       # Welch t검정 (이분산)
stat, p = stats.ttest_rel(before, after)                  # 대응표본 t검정
stat, p = stats.mannwhitneyu(g1, g2, alternative='two-sided')  # 비모수 (Mann-Whitney)
stat, p = stats.wilcoxon(before, after)                   # 비모수 대응표본

# ── ANOVA ───────────────────────────────────
stat, p = stats.f_oneway(g1, g2, g3)    # 일원 ANOVA
stat, p = stats.kruskal(g1, g2, g3)     # 비모수 ANOVA (Kruskal-Wallis)

# 사후검정 (ANOVA 유의 시)
from statsmodels.stats.multicomp import pairwise_tukeyhsd
tukey = pairwise_tukeyhsd(df['값'], df['그룹'], alpha=0.05)
print(tukey.summary())

# ── 상관분석 ────────────────────────────────
r, p = stats.pearsonr(x, y)     # 피어슨 (연속형, 정규분포)
r, p = stats.spearmanr(x, y)    # 스피어만 (순위형, 비정규)
r, p = stats.kendalltau(x, y)   # 켄달 (소표본, 이상치 강건)
# r: 상관계수, p: 유의확률

# ── 카이제곱 검정 ────────────────────────────
from scipy.stats import chi2_contingency, fisher_exact
ct = pd.crosstab(df['변수1'], df['변수2'])
chi2, p, dof, expected = chi2_contingency(ct)
# 셀 기대빈도 5미만 많을 때 → Fisher 정확 검정
odds, p = fisher_exact(ct)  # 2x2 테이블만 가능

# ── 효과크기 ────────────────────────────────
# Cohen's d (두 집단 평균 차이)
def cohens_d(g1, g2):
    n1, n2 = len(g1), len(g2)
    pooled_std = np.sqrt(((n1-1)*g1.std()**2 + (n2-1)*g2.std()**2) / (n1+n2-2))
    return (g1.mean() - g2.mean()) / pooled_std
# 0.2: 소, 0.5: 중, 0.8: 대

# Cramér's V (카이제곱 효과크기)
def cramers_v(chi2, n, k, r):
    return np.sqrt(chi2 / (n * (min(k, r) - 1)))

# ── 결과 해석 출력 ──────────────────────────
print(f'통계량: {stat:.4f}, p-value: {p:.4f}')
print('유의함 (p<0.05)' if p < 0.05 else '유의하지 않음')

# ── 신뢰구간 ────────────────────────────────
ci = stats.t.interval(0.95, df=len(data)-1,
                       loc=np.mean(data),
                       scale=stats.sem(data))
print(f'95% 신뢰구간: {ci[0]:.4f} ~ {ci[1]:.4f}')

---
# 5️⃣ 인과추론 (PSM / IPTW / OW)

In [ ]:
# pip install causalinference zepid
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import statsmodels.formula.api as smf

# ── 공통: 성향점수 추정 ─────────────────────
# treatment: 처치 여부 (0/1)
# covariates: 교란변수들
X = df[['공변량1', '공변량2', '공변량3']]
T = df['treatment']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

lr = LogisticRegression(max_iter=1000)
lr.fit(X_scaled, T)
df['ps'] = lr.predict_proba(X_scaled)[:, 1]  # 성향점수 (propensity score)

print(f'성향점수 범위: {df["ps"].min():.4f} ~ {df["ps"].max():.4f}')

# ── 공통지지 확인 (처치군/통제군 성향점수 분포) ─
import matplotlib.pyplot as plt
plt.figure(figsize=(8,4))
df[df['treatment']==1]['ps'].hist(bins=30, alpha=0.5, label='처치군', color='red')
df[df['treatment']==0]['ps'].hist(bins=30, alpha=0.5, label='통제군', color='blue')
plt.xlabel('성향점수')
plt.legend()
plt.title('성향점수 분포 (공통지지 확인)')
plt.show()

In [ ]:
# ── PSM (성향점수 매칭) ─────────────────────
# pip install psmpy
from psmpy import PsmPy
from psmpy.functions import cohenD

psm = PsmPy(df, treatment='treatment', indx='id', exclude=[])
psm.logistic_ps(balance=True)
psm.knn_matched(matcher='propensity_score', replacement=False, caliper=0.05)  # caliper: 허용 범위

# 매칭 후 결과
matched_df = psm.matched_ids  # 매칭된 ID
psm.plot_match(Title='매칭 전후 성향점수 분포', Ylabel='빈도', Xlabel='성향점수')

# 매칭 후 균형 확인 (SMD: Standardized Mean Difference, <0.1이면 양호)
psm.effect_size_plot()

In [ ]:
# ── IPTW (역확률가중치) ─────────────────────
# ATE (Average Treatment Effect): 전체 대상 평균 처치효과
df['weight_ate'] = np.where(
    df['treatment'] == 1,
    1 / df['ps'],           # 처치군: 1/ps
    1 / (1 - df['ps'])      # 통제군: 1/(1-ps)
)

# ATT (Average Treatment Effect on Treated): 처치군 평균 처치효과
df['weight_att'] = np.where(
    df['treatment'] == 1,
    1,                            # 처치군: 1
    df['ps'] / (1 - df['ps'])     # 통제군: ps/(1-ps)
)

# 극단적 가중치 절단 (Trimming) - 안정성 향상
w_99 = df['weight_ate'].quantile(0.99)
df['weight_ate_trim'] = df['weight_ate'].clip(upper=w_99)

# IPTW 가중 회귀분석
import statsmodels.api as sm
model = sm.WLS(df['결과변수'],
               sm.add_constant(df[['treatment', '공변량1', '공변량2']]),
               weights=df['weight_ate_trim'])
result = model.fit()
print(result.summary())

In [ ]:
# ── OW (겹침가중치, Overlap Weighting) ──────
# 공통지지 영역(ps ≈ 0.5)에 더 큰 가중치 부여
# 극단적 성향점수에 강건 → ATE보다 안정적

df['weight_ow'] = np.where(
    df['treatment'] == 1,
    1 - df['ps'],           # 처치군: (1-ps)
    df['ps']                # 통제군: ps
)

# OW 가중 평균 처치효과 (ATO)
treated_ow = (df[df['treatment']==1]['결과변수'] * df[df['treatment']==1]['weight_ow']).sum()
control_ow = (df[df['treatment']==0]['결과변수'] * df[df['treatment']==0]['weight_ow']).sum()
w_treated = df[df['treatment']==1]['weight_ow'].sum()
w_control = df[df['treatment']==0]['weight_ow'].sum()

ATO = (treated_ow / w_treated) - (control_ow / w_control)
print(f'OW 기반 ATO: {ATO:.4f}')

# ── 가중치 방법 비교 요약 ───────────────────
# ATE (IPTW): 전체 모집단 처치효과 추정, 극단적 ps에 불안정
# ATT (IPTW): 처치군과 유사한 통제군 구성
# ATO (OW) : 공통지지 영역 강조, 극단적 ps에 강건, 최근 많이 쓰임

---
# 6️⃣ Scikit-learn

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, roc_auc_score, roc_curve,
                              mean_squared_error, r2_score, mean_absolute_error)
import numpy as np

# ── 전처리 ──────────────────────────────────
# 인코딩
le = LabelEncoder()
df['enc'] = le.fit_transform(df['카테고리'])     # 라벨 인코딩
dummies = pd.get_dummies(df['카테고리'], drop_first=True)  # 원핫 인코딩

# 스케일링
scaler = StandardScaler()   # 평균 0, 표준편차 1
# scaler = MinMaxScaler()   # 0~1 범위
X_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # 주의: test는 transform만!

# 이상치 제거 (IQR 방법)
Q1 = df['값'].quantile(0.25)
Q3 = df['값'].quantile(0.75)
IQR = Q3 - Q1
df_clean = df[(df['값'] >= Q1 - 1.5*IQR) & (df['값'] <= Q3 + 1.5*IQR)]

# ── 데이터 분리 ─────────────────────────────
X = df.drop(columns=['타겟'])
y = df['타겟']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y  # 클래스 비율 유지
)

# ── 클러스터링 (K-Means) ─────────────────────
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# 최적 k 찾기
inertias, silhouettes = [], []
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(range(2,11), inertias, marker='o')
axes[0].set_title('Elbow Method')
axes[1].plot(range(2,11), silhouettes, marker='o', color='green')
axes[1].set_title('Silhouette Score')
plt.show()

# K-Means 적용
km = KMeans(n_clusters=3, random_state=42, n_init=10)
df['cluster'] = km.fit_predict(X_scaled)

# 클러스터별 특성 확인
df.groupby('cluster')[X.columns.tolist()].mean()

# ── 분류 모델 ───────────────────────────────
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(f'정확도: {accuracy_score(y_test, y_pred):.4f}')
print(f'AUC: {roc_auc_score(y_test, y_prob):.4f}')
print(classification_report(y_test, y_pred))

# ROC 커브
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.plot(fpr, tpr, label=f'AUC={roc_auc_score(y_test, y_prob):.3f}')
plt.plot([0,1],[0,1], 'k--')
plt.xlabel('FPR'); plt.ylabel('TPR')
plt.legend(); plt.title('ROC Curve')
plt.show()

# 혼동 행렬
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix'); plt.show()

# ── 회귀 모델 ───────────────────────────────
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f'R²: {r2_score(y_test, y_pred):.4f}')
print(f'MAE: {mean_absolute_error(y_test, y_pred):.4f}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}')

# ── 하이퍼파라미터 튜닝 ─────────────────────
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}
gs = GridSearchCV(RandomForestClassifier(random_state=42),
                  param_grid, cv=5, scoring='roc_auc', n_jobs=-1)
gs.fit(X_train, y_train)
print('최적 파라미터:', gs.best_params_)

# ── 특성 중요도 ─────────────────────────────
importances = pd.Series(model.feature_importances_, index=X.columns)
importances.sort_values(ascending=True).plot(kind='barh')
plt.title('Feature Importance'); plt.tight_layout(); plt.show()

---
# 7️⃣ Folium (지도 시각화)

In [ ]:
import folium
from folium.plugins import MarkerCluster, HeatMap, TimestampedGeoJson

# ── 기본 지도 생성 ──────────────────────────
m = folium.Map(
    location=[37.5665, 126.9780],
    zoom_start=12,
    tiles='CartoDB positron'
    # tiles: 'OpenStreetMap' / 'CartoDB positron' / 'CartoDB dark_matter' / 'Stamen Terrain'
)

# ── 마커 추가 ───────────────────────────────
folium.Marker(
    location=[37.5665, 126.9780],
    popup=folium.Popup('<b>서울시청</b><br>클릭 팝업', max_width=200),
    tooltip='마우스오버 텍스트',
    icon=folium.Icon(color='red', icon='home', prefix='fa')  # FontAwesome 아이콘
).add_to(m)

# ── 원형 마커 ───────────────────────────────
folium.CircleMarker(
    location=[37.5665, 126.9780],
    radius=df['값'] / 100,  # 값에 비례한 크기
    color='blue',
    fill=True,
    fill_opacity=0.6,
    popup='원형마커'
).add_to(m)

# ── 데이터프레임으로 마커 여러 개 ───────────
for idx, row in df.iterrows():
    folium.Marker(
        location=[row['위도'], row['경도']],
        popup=f"{row['이름']}: {row['값']}",
        icon=folium.Icon(color='blue' if row['값'] > 0 else 'red')  # 조건부 색상
    ).add_to(m)

# ── 클러스터 마커 (데이터 많을 때) ──────────
mc = MarkerCluster()
for idx, row in df.iterrows():
    folium.Marker([row['위도'], row['경도']], popup=row['이름']).add_to(mc)
mc.add_to(m)

# ── 히트맵 ──────────────────────────────────
heat_data = df[['위도', '경도', '값']].values.tolist()
HeatMap(heat_data, radius=15, blur=10, min_opacity=0.3).add_to(m)

# ── 코로플레스 (단계구분도) ──────────────────
folium.Choropleth(
    geo_data='korea_sido.geojson',
    data=df,
    columns=['지역명', '값'],
    key_on='feature.properties.name',
    fill_color='YlOrRd',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='범례 이름',
    nan_fill_color='white'
).add_to(m)

# ── 레이어 컨트롤 ───────────────────────────
folium.LayerControl().add_to(m)

# ── 저장 및 출력 ────────────────────────────
m.save('map.html')
m  # 주피터에서 바로 출력

# ── Streamlit 연동 ───────────────────────────
# pip install streamlit-folium
# from streamlit_folium import st_folium
# st_folium(m, width=900, height=500)

---
# 8️⃣ 현업 자주 쓰는 패턴 모음

In [ ]:
import pandas as pd
import numpy as np

# ── EDA 자동화 ──────────────────────────────
# pip install ydata-profiling
from ydata_profiling import ProfileReport
profile = ProfileReport(df, title='EDA 리포트')
profile.to_file('eda_report.html')

# ── 다중공선성 확인 (VIF) ────────────────────
from statsmodels.stats.outliers_influence import variance_inflation_factor
X_vif = df[['변수1', '변수2', '변수3']]
vif = pd.DataFrame()
vif['변수'] = X_vif.columns
vif['VIF'] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
print(vif)  # VIF > 10 이면 다중공선성 의심

# ── 회귀분석 (statsmodels, p값 확인용) ──────
import statsmodels.formula.api as smf
model = smf.ols('결과변수 ~ 처치 + 공변량1 + 공변량2', data=df).fit()
print(model.summary())
# 로지스틱 회귀
model = smf.logit('이진결과 ~ 처치 + 공변량1', data=df).fit()
print(model.summary())
# 오즈비
print(np.exp(model.params))  # OR
print(np.exp(model.conf_int()))  # 95% CI

# ── 표준화 계수 (변수 중요도 비교) ──────────
from sklearn.preprocessing import StandardScaler
X_std = StandardScaler().fit_transform(df[['변수1', '변수2', '변수3']])
X_std_df = pd.DataFrame(X_std, columns=['변수1', '변수2', '변수3'])

# ── 날짜 관련 자주 쓰는 패턴 ────────────────
df['날짜'] = pd.to_datetime(df['날짜'])
df['연월'] = df['날짜'].dt.to_period('M')           # 연월 (2024-01)
df['주차'] = df['날짜'].dt.isocalendar().week        # 주차
df['분기'] = df['날짜'].dt.quarter                   # 분기
df['주말여부'] = df['날짜'].dt.dayofweek >= 5        # True/False

# 날짜 필터링
df[df['날짜'] >= '2024-01-01']
df[df['날짜'].between('2024-01-01', '2024-12-31')]

# ── 코호트 분석 기본 뼈대 ────────────────────
df['코호트'] = df.groupby('유저ID')['날짜'].transform('min').dt.to_period('M')
df['경과월'] = (df['날짜'].dt.to_period('M') - df['코호트']).apply(lambda x: x.n)
cohort = df.groupby(['코호트', '경과월'])['유저ID'].nunique().unstack()
cohort_rate = cohort.divide(cohort[0], axis=0)  # 리텐션율
sns.heatmap(cohort_rate, annot=True, fmt='.0%', cmap='YlGnBu')

# ── 리텐션 / 이탈 지표 ───────────────────────
# MAU, DAU
MAU = df.groupby(df['날짜'].dt.to_period('M'))['유저ID'].nunique()
DAU = df.groupby(df['날짜'].dt.date)['유저ID'].nunique()

# ── 중복 제거 후 최신 레코드만 ──────────────
df_latest = df.sort_values('날짜').drop_duplicates(subset=['유저ID'], keep='last')

# ── 긴 포맷 ↔ 넓은 포맷 변환 ────────────────
# 긴 → 넓은
df_wide = df.pivot_table(index='유저ID', columns='이벤트', values='횟수', fill_value=0)
# 넓은 → 긴
df_long = df_wide.reset_index().melt(id_vars='유저ID', var_name='이벤트', value_name='횟수')

# ── 샘플링 ──────────────────────────────────
df_sample = df.sample(n=1000, random_state=42)          # 랜덤 1000개
df_sample = df.sample(frac=0.1, random_state=42)        # 10% 샘플링
# 층화 샘플링
df_strat = df.groupby('그룹', group_keys=False).apply(
    lambda x: x.sample(min(len(x), 100), random_state=42)
)